# BloodMNIST + OrganCMNIST — $z_{\mathrm{combined}}$ vs $H$

Cross-dataset generalization notebook (secondary to DermaMNIST).

For each dataset:
1. **Train** ResNet-18 + associative memory (same protocol as `dermamnist_full_validation.ipynb` Part I)
2. **Deploy** label-free memory features on ID calibration + ID test
3. **Evaluate** $z_{\mathrm{combined}}$ vs normalized entropy $H$ on held-out test (AUROC, AUPRC)

**Sample caps (match DermaMNIST):** train $=6408$, cal $=1602$, test $=2005$; probe eval uses $n=2000$ test subsample per seed.

Probe fitting uses ID calibration only; test labels are used only for offline metrics.

## 0. Setup

In [1]:
from __future__ import annotations

import json
import pickle
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def find_repo_root(cwd: Path | None = None) -> Path:
    root = Path(cwd or Path.cwd()).resolve()
    if (root / "research").exists():
        return root
    if (root.parent / "research").exists():
        return root.parent
    return root


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from optimizer.associative_memory import AssociativeMemoryConfig
from research.common.clinical_datasets import ClinicalDatasetConfig, load_clinical_bundle
from research.common.clinical_training import (
    ClinicalTrainingConfig,
    checkpoint_path,
    save_checkpoint,
    save_frozen_memory,
    train_one_seed,
)
from research.common.deployment_pipeline import run_deployment_on_split
from research.common.memory import associative_artifacts_exist, load_associative_artifacts

print("imports ok")

imports ok


## Configuration

In [2]:
TASKS = ("bloodmnist", "organcmnist")
OUTPUT_DIR = REPO_ROOT / "research" / "associative_memory_fft" / "cross_dataset"
PROBE_DIR = OUTPUT_DIR / "probe_results"
FEATURE_ROOT = OUTPUT_DIR / "probe_features"
DATA_DIR = REPO_ROOT / "data" / "clinical"
SEEDS = (42, 123, 456)

RUN_BUILD = True
FORCE_RETRAIN = False
FORCE_REDEPLOY = False
LOGISTIC_C = 1.0
RANDOM_SEED = 42

# Match DermaMNIST protocol (dermamnist_full_validation + memory_vector_vs_magnitude_probe)
DERMA_TRAIN = 6408
DERMA_CAL = 1602
DERMA_TEST = 2005
TEST_SAMPLE_SIZE = 2000  # per-seed probe eval subsample (same as DermaMNIST probe)

FAST_MODE = False
ASSOC_CFG = AssociativeMemoryConfig(use_fft=True, use_attention=False)

if FAST_MODE:
    SEEDS = (42,)
    EPOCHS, MAX_TRAIN, MAX_CAL, MAX_TEST, MAX_EXT = 2, 800, 200, 400, 400
else:
    EPOCHS = 8
    MAX_TRAIN, MAX_CAL, MAX_TEST = DERMA_TRAIN, DERMA_CAL, DERMA_TEST
    MAX_EXT = DERMA_TEST

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROBE_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_ROOT.mkdir(parents=True, exist_ok=True)

print("TASKS", TASKS)
print("OUTPUT_DIR", OUTPUT_DIR)
print(f"caps: train={MAX_TRAIN} cal={MAX_CAL} test={MAX_TEST} probe_n={TEST_SAMPLE_SIZE}")
print(f"RUN_BUILD={RUN_BUILD}  FORCE_RETRAIN={FORCE_RETRAIN}  FORCE_REDEPLOY={FORCE_REDEPLOY}")

TASKS ('bloodmnist', 'organcmnist')
OUTPUT_DIR C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\associative_memory_fft\cross_dataset
caps: train=6408 cal=1602 test=2005 probe_n=2000
RUN_BUILD=True  FORCE_RETRAIN=False  FORCE_REDEPLOY=False


## Probe helpers

Feature dimension follows `num_classes` per task ($d_v = C$).

In [3]:
def mag_cols() -> list[str]:
    return [f"z{j}_magnitude" for j in range(1, 5)]


def full_vector_cols(num_classes: int) -> list[str]:
    cols: list[str] = []
    for j in range(1, 5):
        cols.extend([f"z{j}_d{d}" for d in range(num_classes)])
    return cols


def feature_cache_paths(task: str, seed: int) -> tuple[Path, Path]:
    seed_dir = FEATURE_ROOT / task / f"seed{seed}"
    return seed_dir / "cal_features.csv", seed_dir / "test_features_full.csv"


def subsample_test_df(test_full: pd.DataFrame, *, sample_size: int | None, seed: int) -> pd.DataFrame:
    if sample_size is None or sample_size >= len(test_full):
        return test_full.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(test_full), size=int(sample_size), replace=False))
    return test_full.iloc[idx].reset_index(drop=True)


def fit_l2_probe_on_cal(cal_df: pd.DataFrame, feature_cols: list[str], *, representation: str) -> dict[str, Any]:
    req = feature_cols + ["error"]
    mask = cal_df[req].notna().all(axis=1).to_numpy()
    x_cal = cal_df.loc[mask, feature_cols].to_numpy(dtype=np.float64)
    y_cal = cal_df.loc[mask, "error"].to_numpy(dtype=int)
    if len(x_cal) < 10 or len(np.unique(y_cal)) < 2:
        raise ValueError(f"Invalid calibration data for {representation}: n={len(x_cal)}")
    pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=LOGISTIC_C, max_iter=5000, random_state=42)),
        ]
    )
    pipe.fit(x_cal, y_cal)
    cal_scores = pipe.predict_proba(x_cal)[:, 1]
    return {
        "representation": representation,
        "feature_cols": feature_cols,
        "n_cal": int(len(x_cal)),
        "intercept": float(pipe.named_steps["clf"].intercept_[0]),
        "pipeline": pipe,
        "cal_auroc": float(roc_auc_score(y_cal, cal_scores)),
        "cal_auprc": float(average_precision_score(y_cal, cal_scores)),
    }


def eval_l2_probe(fitted: dict[str, Any], df: pd.DataFrame) -> dict[str, Any]:
    feature_cols = fitted["feature_cols"]
    req = feature_cols + ["error"]
    mask = df[req].notna().all(axis=1).to_numpy()
    x = df.loc[mask, feature_cols].to_numpy(dtype=np.float64)
    y = df.loc[mask, "error"].to_numpy(dtype=int)
    scores = fitted["pipeline"].predict_proba(x)[:, 1]
    return {
        **fitted,
        "n_test": int(len(y)),
        "auroc": float(roc_auc_score(y, scores)),
        "auprc": float(average_precision_score(y, scores)),
    }


def eval_entropy_baseline(test_df: pd.DataFrame) -> dict[str, float]:
    mask = test_df[["normalized_entropy", "error"]].notna().all(axis=1).to_numpy()
    y = test_df.loc[mask, "error"].to_numpy(dtype=int)
    h = test_df.loc[mask, "normalized_entropy"].to_numpy(dtype=float)
    if len(y) < 10 or len(np.unique(y)) < 2:
        return {"n_test": int(len(y)), "auroc": float("nan"), "auprc": float("nan")}
    return {
        "n_test": int(len(y)),
        "auroc": float(roc_auc_score(y, h)),
        "auprc": float(average_precision_score(y, h)),
    }

## 1. Train (Part I — per task)

Mirrors `dermamnist_full_validation.ipynb` §1. Skips seeds with existing checkpoints unless `FORCE_RETRAIN=True`.

In [4]:
bundles: dict[str, Any] = {}
train_summaries: list[dict[str, Any]] = []

train_cfg = ClinicalTrainingConfig(
    seeds=SEEDS,
    epochs=EPOCHS,
    output_dir=OUTPUT_DIR,
    verbose=1,
    associative_memory=ASSOC_CFG,
)

for task in TASKS:
    ds_cfg = ClinicalDatasetConfig(
        task=task,
        data_dir=DATA_DIR,
        max_train=MAX_TRAIN,
        max_cal=MAX_CAL,
        max_test=MAX_TEST,
        max_external=MAX_EXT,
    )
    bundle = load_clinical_bundle(ds_cfg)
    bundles[task] = bundle
    print(f"{task}: n_train={len(bundle.x_train)} n_cal={len(bundle.x_cal)} n_test={len(bundle.x_test)} C={bundle.num_classes}")

    if not RUN_BUILD:
        missing = [s for s in SEEDS if not associative_artifacts_exist(OUTPUT_DIR, task, s)]
        if missing:
            raise FileNotFoundError(f"{task}: missing artifacts for seeds {missing}. Set RUN_BUILD=True.")
        print(f"  RUN_BUILD=False — using cached artifacts for {task}")
        continue

    for seed in SEEDS:
        ckpt = checkpoint_path(OUTPUT_DIR, task, seed)
        if FORCE_RETRAIN and ckpt.exists():
            ckpt.unlink(missing_ok=True)
        if not FORCE_RETRAIN and ckpt.exists() and associative_artifacts_exist(OUTPUT_DIR, task, seed):
            print(f"  skip {task} seed={seed} (checkpoint + associative artifacts exist)")
            continue
        params, opt_state, _, _, summary = train_one_seed(bundle, seed=seed, cfg=train_cfg)
        save_checkpoint(ckpt, params=params, opt_state=opt_state, task=task, seed=seed, cfg=train_cfg, summary=summary)
        save_frozen_memory(OUTPUT_DIR / "memory" / f"{task}_seed{seed}_memory.npz", opt_state, task=task, seed=seed)
        train_summaries.append(summary)
        print(f"  trained {task} seed={seed} test_acc={summary.get('test_acc'):.3f}")

bloodmnist: n_train=6408 n_cal=1602 n_test=2005 C=8
  [bloodmnist seed=42] training: 6408 samples, 8 epochs, ~101 batches/epoch
  [bloodmnist seed=42] epoch 1/8: train_loss=0.6775 cal_acc=0.816 test_acc=0.831 (623.4s)
  [bloodmnist seed=42] epoch 2/8: train_loss=0.4083 cal_acc=0.858 test_acc=0.865 (591.8s)
  [bloodmnist seed=42] epoch 3/8: train_loss=0.3351 cal_acc=0.853 test_acc=0.853 (604.6s)
  [bloodmnist seed=42] epoch 4/8: train_loss=0.3102 cal_acc=0.833 test_acc=0.834 (553.5s)
  [bloodmnist seed=42] epoch 5/8: train_loss=0.2500 cal_acc=0.866 test_acc=0.875 (578.4s)
  [bloodmnist seed=42] epoch 6/8: train_loss=0.2362 cal_acc=0.890 test_acc=0.892 (575.8s)
  [bloodmnist seed=42] epoch 7/8: train_loss=0.2074 cal_acc=0.898 test_acc=0.897 (572.4s)
  [bloodmnist seed=42] epoch 8/8: train_loss=0.1488 cal_acc=0.900 test_acc=0.899 (795.3s)
  trained bloodmnist seed=42 test_acc=0.899
  [bloodmnist seed=123] training: 6408 samples, 8 epochs, ~101 batches/epoch
  [bloodmnist seed=123] epoch 1

## 2. Deploy features + $z_{\mathrm{combined}}$ vs $H$

Label-free deployment on cal/test; logistic probe fit on cal only. $X_{\mathrm{best}}$ chosen by mean calibration AUROC (magnitude vs full vector).

In [5]:
def deploy_split_for_seed(
    task: str,
    seed: int,
    x: np.ndarray,
    y: np.ndarray,
    ids: np.ndarray,
    *,
    num_classes: int,
) -> pd.DataFrame:
    if not associative_artifacts_exist(OUTPUT_DIR, task, seed):
        raise FileNotFoundError(f"Missing associative artifacts for {task} seed={seed}")
    with open(checkpoint_path(OUTPUT_DIR, task, seed), "rb") as f:
        params = pickle.load(f)["params"]
    mem_state, checkpoints, _ = load_associative_artifacts(OUTPUT_DIR, task, seed)
    _, df = run_deployment_on_split(
        params,
        x,
        y,
        ids,
        mem_state,
        checkpoints,
        num_classes=num_classes,
        include_history=False,
    )
    return df


def load_or_deploy_features(task: str, bundle: Any, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    cal_path, test_path = feature_cache_paths(task, seed)
    if not FORCE_REDEPLOY and cal_path.exists() and test_path.exists():
        return pd.read_csv(cal_path), pd.read_csv(test_path)

    print(f"  deploying {task} seed={seed} ...", flush=True)
    cal_df = deploy_split_for_seed(
        task,
        seed,
        bundle.x_cal,
        bundle.y_cal,
        bundle.sample_ids["cal"],
        num_classes=bundle.num_classes,
    )
    test_full = deploy_split_for_seed(
        task,
        seed,
        bundle.x_test,
        bundle.y_test,
        bundle.sample_ids["test"],
        num_classes=bundle.num_classes,
    )
    cal_path.parent.mkdir(parents=True, exist_ok=True)
    cal_df.to_csv(cal_path, index=False)
    test_full.to_csv(test_path, index=False)
    return cal_df, test_full


phase1_rows: list[dict[str, Any]] = []
phase3_rows: list[dict[str, Any]] = []
task_summaries: dict[str, dict[str, Any]] = {}

for task in TASKS:
    bundle = bundles[task]
    mag_feature_cols = mag_cols()
    full_feature_cols = full_vector_cols(bundle.num_classes)
    cal_frames: dict[int, pd.DataFrame] = {}
    test_frames: dict[int, pd.DataFrame] = {}

    for seed in SEEDS:
        cal_df, test_full = load_or_deploy_features(task, bundle, seed)
        cal_frames[seed] = cal_df
        test_frames[seed] = subsample_test_df(
            test_full,
            sample_size=TEST_SAMPLE_SIZE,
            seed=RANDOM_SEED + seed,
        )

    for seed in SEEDS:
        cal_df = cal_frames[seed]
        test_df = test_frames[seed]
        mag_fit = fit_l2_probe_on_cal(cal_df, mag_feature_cols, representation="magnitude")
        full_fit = fit_l2_probe_on_cal(cal_df, full_feature_cols, representation="full_vector")
        mag_cal = eval_l2_probe(mag_fit, cal_df)
        full_cal = eval_l2_probe(full_fit, cal_df)
        mag_test = eval_l2_probe(mag_fit, test_df)
        full_test = eval_l2_probe(full_fit, test_df)
        for rep, cal_run, test_run in (
            ("magnitude", mag_cal, mag_test),
            ("full_vector", full_cal, full_test),
        ):
            phase1_rows.append(
                {
                    "task": task,
                    "seed": seed,
                    "representation": rep,
                    "n_cal": cal_run["n_cal"],
                    "n_test": test_run["n_test"],
                    "cal_auroc": cal_run["auroc"],
                    "test_auroc": test_run["auroc"],
                    "test_auprc": test_run["auprc"],
                }
            )

    task_phase1 = pd.DataFrame([r for r in phase1_rows if r["task"] == task])
    mag_cal_mean = float(task_phase1.loc[task_phase1["representation"] == "magnitude", "cal_auroc"].mean())
    full_cal_mean = float(task_phase1.loc[task_phase1["representation"] == "full_vector", "cal_auroc"].mean())
    x_best_name = "full_vector" if full_cal_mean >= mag_cal_mean else "magnitude"
    x_best_cols = full_feature_cols if x_best_name == "full_vector" else mag_feature_cols

    for seed in SEEDS:
        cal_df = cal_frames[seed]
        test_df = test_frames[seed]
        z_fit = fit_l2_probe_on_cal(cal_df, x_best_cols, representation=f"z_combined_{x_best_name}")
        z_test = eval_l2_probe(z_fit, test_df)
        h_test = eval_entropy_baseline(test_df)
        phase3_rows.append(
            {
                "task": task,
                "seed": seed,
                "model": "entropy_H",
                "representation": x_best_name,
                "n_cal": z_fit["n_cal"],
                "n_test": h_test["n_test"],
                "auroc": h_test["auroc"],
                "auprc": h_test["auprc"],
            }
        )
        phase3_rows.append(
            {
                "task": task,
                "seed": seed,
                "model": "z_combined",
                "representation": x_best_name,
                "n_cal": z_fit["n_cal"],
                "n_test": z_test["n_test"],
                "auroc": z_test["auroc"],
                "auprc": z_test["auprc"],
            }
        )
        phase3_rows.append(
            {
                "task": task,
                "seed": seed,
                "model": "delta_z_combined_minus_H",
                "representation": x_best_name,
                "n_cal": z_fit["n_cal"],
                "n_test": z_test["n_test"],
                "auroc": float(z_test["auroc"] - h_test["auroc"]),
                "auprc": float(z_test["auprc"] - h_test["auprc"]),
            }
        )

    task_phase3 = pd.DataFrame([r for r in phase3_rows if r["task"] == task])
    h_auroc_mean = float(task_phase3.loc[task_phase3["model"] == "entropy_H", "auroc"].mean())
    z_auroc_mean = float(task_phase3.loc[task_phase3["model"] == "z_combined", "auroc"].mean())
    h_auprc_mean = float(task_phase3.loc[task_phase3["model"] == "entropy_H", "auprc"].mean())
    z_auprc_mean = float(task_phase3.loc[task_phase3["model"] == "z_combined", "auprc"].mean())
    task_summaries[task] = {
        "task": task,
        "num_classes": bundle.num_classes,
        "x_best": x_best_name,
        "mean_auroc_H": h_auroc_mean,
        "mean_auroc_z_combined": z_auroc_mean,
        "mean_delta_auroc": float(z_auroc_mean - h_auroc_mean),
        "mean_auprc_H": h_auprc_mean,
        "mean_auprc_z_combined": z_auprc_mean,
        "mean_delta_auprc": float(z_auprc_mean - h_auprc_mean),
        "z_combined_beats_H_on_auroc": z_auroc_mean > h_auroc_mean,
        "z_combined_beats_H_on_auprc": z_auprc_mean > h_auprc_mean,
    }

phase1_df = pd.DataFrame(phase1_rows)
phase3_df = pd.DataFrame(phase3_rows)
summary_df = pd.DataFrame(task_summaries.values())

phase1_df.to_csv(PROBE_DIR / "phase1_per_task_seed.csv", index=False)
phase3_df.to_csv(PROBE_DIR / "phase3_per_task_seed.csv", index=False)
summary_df.to_csv(PROBE_DIR / "cross_dataset_probe_summary.csv", index=False)

results = {
    "tasks": list(TASKS),
    "seeds": list(SEEDS),
    "test_sample_size": TEST_SAMPLE_SIZE,
    "task_summaries": task_summaries,
}
(PROBE_DIR / "cross_dataset_probe_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

display(summary_df.round(4))
display(
    phase3_df.pivot_table(
        index=["task", "seed"],
        columns="model",
        values=["auroc", "auprc"],
    ).round(4)
)

  deploying bloodmnist seed=42 ...
  deploying bloodmnist seed=123 ...
  deploying bloodmnist seed=456 ...
  deploying organcmnist seed=42 ...
  deploying organcmnist seed=123 ...
  deploying organcmnist seed=456 ...


,task,num_classes,x_best,mean_auroc_H,mean_auroc_z_combined,mean_delta_auroc,mean_auprc_H,mean_auprc_z_combined,mean_delta_auprc,z_combined_beats_H_on_auroc,z_combined_beats_H_on_auprc
0,bloodmnist,8,full_vector,0.7693,0.8837,0.1144,0.8896,0.9439,0.0543,True,True
1,organcmnist,11,full_vector,0.8243,0.8772,0.0529,0.9256,0.9404,0.0148,True,True


auprc                       \
model            delta_z_combined_minus_H entropy_H z_combined   
task        seed                                                 
bloodmnist  42                     0.0217    0.9606     0.9822   
            123                    0.0493    0.8917     0.9410   
            456                    0.0919    0.8165     0.9084   
organcmnist 42                    -0.0029    0.9518     0.9489   
            123                    0.0084    0.9526     0.9610   
            456                    0.0388    0.8724     0.9112   

                                    auroc                       
model            delta_z_combined_minus_H entropy_H z_combined  
task        seed                                                
bloodmnist  42                     0.0713    0.8657     0.9371  
            123                    0.1055    0.7410     0.8466  
            456                    0.1664    0.7012     0.8676  
organcmnist 42                     0.0096    0.8826     0.8922  
            123                    0.0384    0.8705     0.9088  
            456                    0.1108    0.7198     0.8306

## 3. Summary

In [7]:
summary_path = PROBE_DIR / "cross_dataset_probe_summary.md"
lines = ["# BloodMNIST + OrganCMNIST — $z_{\\mathrm{combined}}$ vs $H$", ""]
for task, s in task_summaries.items():
    lines.append(
        f"- **{task}** ($X_{{\\mathrm{{best}}}}={s['x_best']}$): "
        f"AUROC $z_{{\\mathrm{{combined}}}}$={s['mean_auroc_z_combined']:.4f} vs $H$={s['mean_auroc_H']:.4f} "
        f"($\\Delta$={s['mean_delta_auroc']:+.4f}); "
        f"AUPRC {s['mean_auprc_z_combined']:.4f} vs {s['mean_auprc_H']:.4f} "
        f"($\\Delta$={s['mean_delta_auprc']:+.4f})"
    )
summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(summary_path.read_text(encoding="utf-8"))

# BloodMNIST + OrganCMNIST — $z_{\mathrm{combined}}$ vs $H$

- **bloodmnist** ($X_{\mathrm{best}}=full_vector$): AUROC $z_{\mathrm{combined}}$=0.8837 vs $H$=0.7693 ($\Delta$=+0.1144); AUPRC 0.9439 vs 0.8896 ($\Delta$=+0.0543)
- **organcmnist** ($X_{\mathrm{best}}=full_vector$): AUROC $z_{\mathrm{combined}}$=0.8772 vs $H$=0.8243 ($\Delta$=+0.0529); AUPRC 0.9404 vs 0.9256 ($\Delta$=+0.0148)

